# 22 OTM Model-Ready Cleanup

Final cleanup pass before modeling.

Goals:
- normalize leftover non-Latin or awkward display names
- remove obvious low-value tourism noise
- keep a traceable record of what was removed and why


In [1]:
import re

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)


In [2]:
INPUT_PATH = "../data/processed/otm_pois_with_wikipedia.csv"
MODEL_READY_PATH = "../data/processed/otm_pois_model_ready.csv"
REMOVED_PATH = "../data/processed/otm_pois_removed_noise.csv"

df = pd.read_csv(INPUT_PATH)
print("Input shape:", df.shape)
df[["display_name_en", "category_clean", "kinds", "wiki_title", "wiki_has_page"]].head(10)


Input shape: (327, 27)


,display_name_en,category_clean,kinds,wiki_title,wiki_has_page
0,Chora Mosque / Kariye Museum,museum,"religion,mosques,churches,cultural,museums,int...",Chora Church,1
1,Hagia Sophia,museum,"religion,mosques,churches,cultural,museums,int...",Hagia Sophia,1
2,Serpent Column,historic,"historic,monuments_and_memorials,burial_places...",Serpent Column,1
3,Süleymaniye Mosque,religious,"religion,mosques,interesting_places",Süleymaniye Mosque,1
4,The Blue Mosque,religious,"religion,mosques,interesting_places",Sultan Ahmed Mosque,1
5,Tomb of Sultan Ahmet,religious,"religion,mosques,interesting_places",Sultan Ahmed Mosque,1
6,Yıldız Palace,museum,"palaces,architecture,historic_architecture,cul...",Yıldız Palace,1
7,15 July coup monument (Istanbul),historic,"historic,monuments_and_memorials,interesting_p...",Timeline of Istanbul,1
8,Abbas Ağa Fountain,historic,"fountains,historic,cultural,urban_environment,...",NaN,0
9,Adam Mickiewicz Museum,museum,"biographical_museums,historic_house_museums,cu...",Adam Mickiewicz Museum,1


## Helpers

In [3]:
NON_LATIN_RE = re.compile(r"[\u0400-\u04FF\u0600-\u06FF\u0370-\u03FF]")

FOUNTAIN_KEEP = {
    "German Fountain",
    "Sultan Ahmed III Fountain",
    "Fountain of Ahmed III (Üsküdar)",
}

DROP_NAME_PATTERNS = [
    (re.compile(r"botanical garden", re.I), "botanical_garden"),
    (re.compile(r"\bpark\b|parkı", re.I), "park"),
    (re.compile(r"\bsquare\b", re.I), "square"),
    (re.compile(r"\bstadium\b", re.I), "stadium"),
]

DROP_KIND_PATTERNS = [
    (re.compile(r"gardens_and_parks", re.I), "parks_and_gardens"),
    (re.compile(r"\bsquares\b", re.I), "square"),
    (re.compile(r"\bstadiums\b", re.I), "stadium"),
]

FOOD_KIND_RE = re.compile(r"restaurants|foods|cafes|coffee|nightclubs|adult", re.I)
FOUNTAIN_RE = re.compile(r"fountain", re.I)


def clean_text(value):
    if pd.isna(value):
        return ""
    return str(value).strip()


def has_non_latin(text):
    return bool(NON_LATIN_RE.search(clean_text(text)))


def choose_model_display_name(row):
    display_name = clean_text(row.get("display_name_en"))
    source_name = clean_text(row.get("name"))
    wiki_title = clean_text(row.get("wiki_title"))

    if not display_name and wiki_title:
        return wiki_title
    if has_non_latin(display_name) and wiki_title:
        return wiki_title
    if has_non_latin(display_name) and source_name and not has_non_latin(source_name):
        return source_name
    return display_name or source_name or wiki_title


def classify_drop_reason(row):
    name = clean_text(row.get("display_name_model"))
    kinds = clean_text(row.get("kinds"))
    category = clean_text(row.get("category_clean"))
    wiki_has_page = str(row.get("wiki_has_page", "0")) == "1"

    if has_non_latin(name) and not wiki_has_page:
        return "non_latin_without_wiki"

    if FOUNTAIN_RE.search(name) and name not in FOUNTAIN_KEEP:
        return "generic_fountain"

    for pattern, reason in DROP_NAME_PATTERNS:
        if pattern.search(name):
            return reason

    for pattern, reason in DROP_KIND_PATTERNS:
        if pattern.search(kinds):
            return reason

    if FOOD_KIND_RE.search(kinds) and category != "museum":
        return "food_or_nightlife_contamination"

    return ""


## Normalize names and flag removals

In [4]:
work_df = df.copy()
work_df["display_name_model"] = work_df.apply(choose_model_display_name, axis=1)
work_df["drop_reason"] = work_df.apply(classify_drop_reason, axis=1)
work_df["model_keep"] = work_df["drop_reason"].eq("")

print("Keep count:", int(work_df["model_keep"].sum()))
print("Drop count:", int((~work_df["model_keep"]).sum()))
work_df["drop_reason"].value_counts(dropna=False)


Keep count: 300
Drop count: 27


drop_reason
                                   300
generic_fountain                     8
park                                 7
stadium                              4
parks_and_gardens                    2
square                               2
non_latin_without_wiki               2
botanical_garden                     1
food_or_nightlife_contamination      1
Name: count, dtype: int64

## Inspect what will be removed

In [5]:
removed_df = work_df.loc[~work_df["model_keep"]].copy()
removed_df[[
    "display_name_en",
    "display_name_model",
    "category_clean",
    "kinds",
    "wiki_title",
    "wiki_has_page",
    "drop_reason",
]].sort_values(["drop_reason", "display_name_model"]).head(50)


,display_name_en,display_name_model,category_clean,kinds,wiki_title,wiki_has_page,drop_reason
13,Alfred Heilbronn Botanical Garden,Alfred Heilbronn Botanical Garden,attraction,"urban_environment,gardens_and_parks,cultural,i...",NaN,0,botanical_garden
311,Tomb of Mimar Sinan,Tomb of Mimar Sinan,historic,"historic,monuments_and_memorials,burial_places...",Mimar Sinan,1,food_or_nightlife_contamination
8,Abbas Ağa Fountain,Abbas Ağa Fountain,historic,"fountains,historic,cultural,urban_environment,...",NaN,0,generic_fountain
95,Hasan Rıza Pasha Fountain,Hasan Rıza Pasha Fountain,historic,"fountains,historic,cultural,urban_environment,...",Hızırbey Mosque Fountain,1,generic_fountain
131,Mısırlı Osman Ağa Fountain,Mısırlı Osman Ağa Fountain,historic,"fountains,historic,cultural,urban_environment,...",Mısırlı Osman Ağa Çeşmesi (Kadıköy),1,generic_fountain
134,Nevşehirli Damat İbrahim Paşa Fountain,Nevşehirli Damat İbrahim Paşa Fountain,historic,"fountains,historic,cultural,urban_environment,...",Nevşehirli Damat Ibrahim Pasha,1,generic_fountain
173,Sineperver Valide Sultan Fountain,Sineperver Valide Sultan Fountain,historic,"fountains,historic,cultural,urban_environment,...",Sineperver Sultan,1,generic_fountain
297,Sultan III. Mustafa Fountain,Sultan III. Mustafa Fountain,historic,"fountains,historic,cultural,urban_environment,...",Beyhan Sultan (daughter of Mustafa III),1,generic_fountain
199,Tophane Fountain,Tophane Fountain,attraction,"fountains,cultural,urban_environment,interesti...",Tophane Fountain,1,generic_fountain
222,Şehzade Numan fountain,Şehzade Numan fountain,historic,"fountains,historic,cultural,urban_environment,...",NaN,0,generic_fountain


## Save outputs

In [6]:
model_ready_df = work_df.loc[work_df["model_keep"]].copy()

model_ready_df["display_name_en"] = model_ready_df["display_name_model"]
model_ready_df = model_ready_df.drop(columns=["display_name_model", "drop_reason", "model_keep"])

removed_df.to_csv(REMOVED_PATH, index=False)
model_ready_df.to_csv(MODEL_READY_PATH, index=False)

print("Saved model-ready:", MODEL_READY_PATH, model_ready_df.shape)
print("Saved removed rows:", REMOVED_PATH, removed_df.shape)
model_ready_df[["display_name_en", "category_clean", "wiki_has_page", "wiki_pageviews_total"]].head(20)


Saved model-ready: ../data/processed/otm_pois_model_ready.csv (300, 27)
Saved removed rows: ../data/processed/otm_pois_removed_noise.csv (27, 30)


,display_name_en,category_clean,wiki_has_page,wiki_pageviews_total
0,Chora Mosque / Kariye Museum,museum,1,20025.0
1,Hagia Sophia,museum,1,1300243.0
2,Serpent Column,historic,1,38432.0
3,Süleymaniye Mosque,religious,1,138681.0
4,The Blue Mosque,religious,1,50304.0
5,Tomb of Sultan Ahmet,religious,1,50304.0
6,Yıldız Palace,museum,1,39930.0
7,15 July coup monument (Istanbul),historic,1,4959.0
9,Adam Mickiewicz Museum,museum,1,68.0
10,Ahi Çelebi Mosque,religious,1,839.0
